# 1. Milk products PCA  
dimension reduction - PCA  
model - Random Forest

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from dash import Dash, dcc, html, Input, Output, Patch, no_update
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor


data = pd.read_csv('data/milk_data.csv', sep=';', index_col=['Product'])
labels = data.index
col_ohe = ['Bacterial Content']
ct = ColumnTransformer(
    [("StandardScaler", StandardScaler(), data.columns.drop(col_ohe).to_list()),
     ("OHE", OneHotEncoder(drop='first'), col_ohe)], verbose_feature_names_out=False)

transformed = pd.DataFrame(data=ct.fit_transform(data), index=labels, columns=ct.get_feature_names_out())


reverse = pd.DataFrame(data=ct.named_transformers_['StandardScaler'].inverse_transform(transformed[list(ct.named_transformers_['StandardScaler'].get_feature_names_out())]),
                       index=labels,
                       columns=list(ct.named_transformers_['StandardScaler'].get_feature_names_out()))
reverse[col_ohe[0]] = ct.named_transformers_['OHE'].inverse_transform(transformed[list(ct.named_transformers_['OHE'].get_feature_names_out())]).reshape(transformed.shape[0])

dict_milk_groups = {'Cottage Cheese': 'sour_cream-cottage', 
                    'Sour Cream': 'sour_cream-cottage',
                    'Matzoni': 'like-matzoni', 
                    'Ryazhenka': 'like_ryazhenka', 
                    'Varenyets': 'like_ryazhenka', 
                    'Yogurt' : 'yogurts', 
                    'Greek Yogurt': 'yogurts',
                    'Kefir': 'kefir_group', 
                    'Koumiss':'kefir_group', 
                    'Brie Cheese':'cheese_group', 
                    'Roquefort Cheese': 'cheese_group', 
                    'Cheddar Cheese': 'cheese_group',
                    'Mozzarella Cheese': 'cheese_group', 
                    'Milk': 'milk', 
                    'Butter': 'butter', 
                    'Dairy Cream': 'dairy_cream', 
                    'Soured Milk':'like-matzoni', 
                    'Buttermilk':'like-matzoni',
                    'Skyr': 'yogurts', 
                    'Ayran': 'like-matzoni', 
                    'Coalhada':'yogurts', 
                    'Dahi': 'yogurts', 
                    'Bryndza Cheese':'cheese_group', 
                    'Suluguni Cheese':'cheese_group', 
                    'Gouda Cheese':'cheese_group',
                    'Parmigiano Cheese':'cheese_group', 
                    'Ice Cream':'ice-cream'}

X_processed = transformed.copy()
reducer = PCA(n_components=2)

Z = reducer.fit_transform(X_processed)

Z = pd.DataFrame(
    Z,
    columns=["component_1", "component_2"]
)

Z["object"] = labels
Z['group'] = Z['object'].map(dict_milk_groups)

X_target = X_processed

reconstruction_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reconstruction_model.fit(
    Z[["component_1", "component_2"]],
    X_target
)



# dash part

# plot
fig = px.scatter(
    Z, x="component_1", y="component_2",
    color = 'group',
    text="object", hover_name="object", title="2D representation"
)
fig.update_traces(marker=dict(size=12), textfont_size=8)

MARKER_IDX = len(fig.data)

# extra trace (index 1) that shows where the user clicked
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(symbol="x", size=16, color="yellow"),
    showlegend=False, hoverinfo="skip"
))
fig.update_layout(uirevision="keep")   # keeps zoom/pan when the figure is patched


# dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H2("Dimension Reduction → Feature Reconstruction"),

    dcc.Store(id="free-click-coordinate"),
    dcc.Store(id="listener-ready"),
    # fires once, shortly after page load, to attach the JS click listener
    dcc.Interval(id="init-interval", interval=500, max_intervals=1),

    html.Div([
        html.Div(
            dcc.Graph(id="reduction-scatter", figure=fig),
            style={"width": "65%", "display": "inline-block", "verticalAlign": "top"}
        ),
        html.Div([
            html.H4("Nearest object"),
            html.Div(id="selected-object", children="Click anywhere on the plot"),
            html.H4("Coordinates"),
            html.Div(id="selected-coordinates"),
            html.H4("Reconstructed features"),
            html.Div(id="reconstructed-features"),
        ], style={"width": "30%", "display": "inline-block",
                  "verticalAlign": "top", "padding": "20px", 'fontSize': '8px'}),
    ])
])


# clientside callback

app.clientside_callback(
    """
    function(n) {
        const wrapper = document.getElementById("reduction-scatter");
        if (!wrapper || wrapper.dataset.freeClick) {
            return window.dash_clientside.no_update;
        }
        wrapper.dataset.freeClick = "1";

        let down = null;
        wrapper.addEventListener("mousedown", e => { down = [e.clientX, e.clientY]; });

        wrapper.addEventListener("click", function(e) {
            // ignore drags (zoom / pan)
            if (down && Math.hypot(e.clientX - down[0], e.clientY - down[1]) > 4) return;

            const gd   = wrapper.querySelector(".js-plotly-plot");
            const drag = wrapper.querySelector(".nsewdrag");   // the inner plot area
            if (!gd || !drag || !gd._fullLayout) return;

            const r = drag.getBoundingClientRect();
            if (e.clientX < r.left || e.clientX > r.right ||
                e.clientY < r.top  || e.clientY > r.bottom) return;   // outside axes

            const x = gd._fullLayout.xaxis.p2d(e.clientX - r.left);
            const y = gd._fullLayout.yaxis.p2d(e.clientY - r.top);

            window.dash_clientside.set_props(
                "free-click-coordinate",
                {data: {x: x, y: y, t: Date.now()}}
            );
        });
        return "ready";
    }
    """,
    Output("listener-ready", "data"),
    Input("init-interval", "n_intervals"),
)


# callback
@app.callback(
    Output("selected-object", "children"),
    Output("selected-coordinates", "children"),
    Output("reconstructed-features", "children"),
    Output("reduction-scatter", "figure"),
    Input("free-click-coordinate", "data"),
    prevent_initial_call=True,
)
def update_reconstruction(click):
    if not click:
        return no_update, no_update, no_update, no_update

    x, y = click["x"], click["y"]

    # nearest real object (just for info)
    d = (Z["component_1"] - x) ** 2 + (Z["component_2"] - y) ** 2
    nearest = Z.loc[d.idxmin(), "object"]

    # reconstruct original features
    #prediction = reconstruction_model.predict([[x, y]])[0]

    prediction = reconstruction_model.predict([[x,y]])[0]
    pred_df = pd.DataFrame(data=[prediction], columns=transformed.columns)
    reverse = pd.DataFrame(data=ct.named_transformers_['StandardScaler'].inverse_transform(pred_df[list(ct.named_transformers_['StandardScaler'].get_feature_names_out())]),    
                           columns=list(ct.named_transformers_['StandardScaler'].get_feature_names_out()))
    reverse[col_ohe[0]] = ct.named_transformers_['OHE'].inverse_transform(pred_df[list(ct.named_transformers_['OHE'].get_feature_names_out())]).reshape(pred_df.shape[0])
    prediction = list(np.array(reverse[:1])[0])

    rows = [
        html.Tr([html.Td(name), html.Td(f"{value:.3f}")])
        for name, value in zip(reverse.columns, prediction)
    ]
    table = html.Table(
        [html.Thead(html.Tr([html.Th("Feature"), html.Th("Estimated value")])),
         html.Tbody(rows)],
        style={"width": "100%"}
    )

    # move the red marker without rebuilding the whole figure
    patched = Patch()
   #  patched["data"][1]["x"] = [x]
   #  patched["data"][1]["y"] = [y]
    patched["data"][MARKER_IDX]["x"] = [x]
    patched["data"][MARKER_IDX]["y"] = [y]

    return (
        nearest,
        html.Div([f"X = {x:.3f}", html.Br(), f"Y = {y:.3f}"]),
        table,
        patched,
    )


# start
if __name__ == "__main__":
    app.run(debug=False, port=8080)

# 2. Milk products UMAP  
dimension reduction - UMAP  
model - KNeighborsRegressor

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from dash import Dash, dcc, html, Input, Output, Patch, no_update
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
import umap
from sklearn.neighbors import KNeighborsRegressor


data = pd.read_csv('data/milk_data.csv', sep=';', index_col=['Product'])
labels = data.index
col_ohe = ['Bacterial Content']
ct = ColumnTransformer(
    [("StandardScaler", StandardScaler(), data.columns.drop(col_ohe).to_list()),
     ("OHE", OneHotEncoder(drop='first'), col_ohe)], verbose_feature_names_out=False)

transformed = pd.DataFrame(data=ct.fit_transform(data), index=labels, columns=ct.get_feature_names_out())


reverse = pd.DataFrame(data=ct.named_transformers_['StandardScaler'].inverse_transform(transformed[list(ct.named_transformers_['StandardScaler'].get_feature_names_out())]),
                       index=labels,
                       columns=list(ct.named_transformers_['StandardScaler'].get_feature_names_out()))
reverse[col_ohe[0]] = ct.named_transformers_['OHE'].inverse_transform(transformed[list(ct.named_transformers_['OHE'].get_feature_names_out())]).reshape(transformed.shape[0])

dict_milk_groups = {'Cottage Cheese': 'sour_cream-cottage', 
                    'Sour Cream': 'sour_cream-cottage',
                    'Matzoni': 'like-matzoni', 
                    'Ryazhenka': 'like_ryazhenka', 
                    'Varenyets': 'like_ryazhenka', 
                    'Yogurt' : 'yogurts', 
                    'Greek Yogurt': 'yogurts',
                    'Kefir': 'kefir_group', 
                    'Koumiss':'kefir_group', 
                    'Brie Cheese':'cheese_group', 
                    'Roquefort Cheese': 'cheese_group', 
                    'Cheddar Cheese': 'cheese_group',
                    'Mozzarella Cheese': 'cheese_group', 
                    'Milk': 'milk', 
                    'Butter': 'butter', 
                    'Dairy Cream': 'dairy_cream', 
                    'Soured Milk':'like-matzoni', 
                    'Buttermilk':'like-matzoni',
                    'Skyr': 'yogurts', 
                    'Ayran': 'like-matzoni', 
                    'Coalhada':'yogurts', 
                    'Dahi': 'yogurts', 
                    'Bryndza Cheese':'cheese_group', 
                    'Suluguni Cheese':'cheese_group', 
                    'Gouda Cheese':'cheese_group',
                    'Parmigiano Cheese':'cheese_group', 
                    'Ice Cream':'ice-cream'}

X_processed = transformed.copy()

reducer = umap.UMAP()
embedding = reducer.fit_transform(X_processed)

Z = pd.DataFrame(
    embedding,
    columns=["component_1", "component_2"]
)

Z["object"] = labels
Z['group'] = Z['object'].map(dict_milk_groups)


X_target = X_processed

reconstruction_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reconstruction_model = KNeighborsRegressor(
    n_neighbors=5, 
    weights='uniform', 
    algorithm='auto', 
    leaf_size=30, 
    p=2, 
    metric='minkowski'
)

reconstruction_model.fit(
    Z[["component_1", "component_2"]],
    X_target
)



# dash part

# plot
fig = px.scatter(
    Z, x="component_1", y="component_2",
    color = 'group',
    text="object", hover_name="object", title="2D representation"
)
fig.update_traces(marker=dict(size=12), textfont_size=8)

MARKER_IDX = len(fig.data)

# extra trace (index 1) that shows where the user clicked
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(symbol="x", size=16, color="yellow"),
    showlegend=False, hoverinfo="skip"
))
fig.update_layout(uirevision="keep")   # keeps zoom/pan when the figure is patched


# dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H2("Dimension Reduction → Feature Reconstruction"),

    dcc.Store(id="free-click-coordinate"),
    dcc.Store(id="listener-ready"),
    # fires once, shortly after page load, to attach the JS click listener
    dcc.Interval(id="init-interval", interval=500, max_intervals=1),

    html.Div([
        html.Div(
            dcc.Graph(id="reduction-scatter", figure=fig),
            style={"width": "65%", "display": "inline-block", "verticalAlign": "top"}
        ),
        html.Div([
            html.H4("Nearest object"),
            html.Div(id="selected-object", children="Click anywhere on the plot"),
            html.H4("Coordinates"),
            html.Div(id="selected-coordinates"),
            html.H4("Reconstructed features"),
            html.Div(id="reconstructed-features"),
        ], style={"width": "30%", "display": "inline-block",
                  "verticalAlign": "top", "padding": "20px", 'fontSize': '8px'}),
    ])
])


# clientside callback
app.clientside_callback(
    """
    function(n) {
        const wrapper = document.getElementById("reduction-scatter");
        if (!wrapper || wrapper.dataset.freeClick) {
            return window.dash_clientside.no_update;
        }
        wrapper.dataset.freeClick = "1";

        let down = null;
        wrapper.addEventListener("mousedown", e => { down = [e.clientX, e.clientY]; });

        wrapper.addEventListener("click", function(e) {
            // ignore drags (zoom / pan)
            if (down && Math.hypot(e.clientX - down[0], e.clientY - down[1]) > 4) return;

            const gd   = wrapper.querySelector(".js-plotly-plot");
            const drag = wrapper.querySelector(".nsewdrag");   // the inner plot area
            if (!gd || !drag || !gd._fullLayout) return;

            const r = drag.getBoundingClientRect();
            if (e.clientX < r.left || e.clientX > r.right ||
                e.clientY < r.top  || e.clientY > r.bottom) return;   // outside axes

            const x = gd._fullLayout.xaxis.p2d(e.clientX - r.left);
            const y = gd._fullLayout.yaxis.p2d(e.clientY - r.top);

            window.dash_clientside.set_props(
                "free-click-coordinate",
                {data: {x: x, y: y, t: Date.now()}}
            );
        });
        return "ready";
    }
    """,
    Output("listener-ready", "data"),
    Input("init-interval", "n_intervals"),
)


# callback
@app.callback(
    Output("selected-object", "children"),
    Output("selected-coordinates", "children"),
    Output("reconstructed-features", "children"),
    Output("reduction-scatter", "figure"),
    Input("free-click-coordinate", "data"),
    prevent_initial_call=True,
)
def update_reconstruction(click):
    if not click:
        return no_update, no_update, no_update, no_update

    x, y = click["x"], click["y"]

    # nearest real object (just for info)
    d = (Z["component_1"] - x) ** 2 + (Z["component_2"] - y) ** 2
    nearest = Z.loc[d.idxmin(), "object"]

    # reconstruct original features
    #prediction = reconstruction_model.predict([[x, y]])[0]

    prediction = reconstruction_model.predict([[x,y]])[0]
    pred_df = pd.DataFrame(data=[prediction], columns=transformed.columns)
    reverse = pd.DataFrame(data=ct.named_transformers_['StandardScaler'].inverse_transform(pred_df[list(ct.named_transformers_['StandardScaler'].get_feature_names_out())]),    
                           columns=list(ct.named_transformers_['StandardScaler'].get_feature_names_out()))
    reverse[col_ohe[0]] = ct.named_transformers_['OHE'].inverse_transform(pred_df[list(ct.named_transformers_['OHE'].get_feature_names_out())]).reshape(pred_df.shape[0])
    prediction = list(np.array(reverse[:1])[0])

    rows = [
        html.Tr([html.Td(name), html.Td(f"{value:.3f}")])
        for name, value in zip(reverse.columns, prediction)
    ]
    table = html.Table(
        [html.Thead(html.Tr([html.Th("Feature"), html.Th("Estimated value")])),
         html.Tbody(rows)],
        style={"width": "100%"}
    )

    # move the red marker without rebuilding the whole figure
    patched = Patch()
   #  patched["data"][1]["x"] = [x]
   #  patched["data"][1]["y"] = [y]
    patched["data"][MARKER_IDX]["x"] = [x]
    patched["data"][MARKER_IDX]["y"] = [y]

    return (
        nearest,
        html.Div([f"X = {x:.3f}", html.Br(), f"Y = {y:.3f}"]),
        table,
        patched,
    )


# start
if __name__ == "__main__":
    app.run(debug=False, port=8080)

/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12

# 3. Alcohol products UMAP  
dimension reduction - UMAP  
model Random Forest

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer
from dash import Dash, dcc, html, Input, Output, Patch, no_update
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
import umap


data = pd.read_excel('data/alcohol_products_table_r.xlsx', index_col=[0])
labels = data.index
col_ohe = ['Естественное брожение', 'Газированность', 'Исходный продукт']
sqrtran = FunctionTransformer(np.sqrt, inverse_func=np.square)
ct = ColumnTransformer(
    [("StandardScaler", StandardScaler(), data.columns.drop(['Естественное брожение', 'Газированность', 'Исходный продукт']).to_list()),
     ("OHE", OneHotEncoder(drop='first'), ['Естественное брожение', 'Газированность', 'Исходный продукт'])], verbose_feature_names_out=False)

transformed = pd.DataFrame(data=ct.fit_transform(data), index=labels, columns=ct.get_feature_names_out())

reverse = pd.DataFrame(data=ct.named_transformers_['StandardScaler'].inverse_transform(transformed[list(ct.named_transformers_['StandardScaler'].get_feature_names_out())]),
                       index=labels,
                       columns=list(ct.named_transformers_['StandardScaler'].get_feature_names_out()))

reverse_ohe = pd.DataFrame(data=ct.named_transformers_['OHE'].inverse_transform(transformed[list(ct.named_transformers_['OHE'].get_feature_names_out())]),
                           index=labels,
                           columns=col_ohe)

for a in reverse_ohe.columns:
    reverse[a] = reverse_ohe[a]

dict_alk_groups = {'Красное вино':'вино',
                   'Белое вино':'вино',
                   'Молодое вино':'вино',
                   'Сидр':'пиво',
                   'Лагер':'пиво',
                   'ИПА':'пиво',
                   'Квас':'пиво',
                   'Самогон':'самогон',
                   'Чача':'самогон',
                   'Граппа':'самогон',
                   'Виски':'самогон',
                   'Текилла':'самогон',
                   'Водка':'водка',
                   'Аквавит':'водка',
                   'Кефир':'кефир'}

X_processed = transformed.copy()

reducer = umap.UMAP()
embedding = reducer.fit_transform(X_processed)


Z = pd.DataFrame(
    embedding,
    columns=["component_1", "component_2"]
)

Z["object"] = labels
Z['group'] = Z['object'].map(dict_alk_groups)

X_target = X_processed

reconstruction_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reconstruction_model.fit(
    Z[["component_1", "component_2"]],
    X_target
)



# dash part

# plot
fig = px.scatter(
    Z, x="component_1", y="component_2",
    color = 'group',
    text="object", hover_name="object", title="2D representation"
)
fig.update_traces(marker=dict(size=12), textfont_size=8)

MARKER_IDX = len(fig.data)

# extra trace (index 1) that shows where the user clicked
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(symbol="x", size=16, color="yellow"),
    showlegend=False, hoverinfo="skip"
))
fig.update_layout(uirevision="keep")   # keeps zoom/pan when the figure is patched


# dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H2("Dimension Reduction → Feature Reconstruction"),

    dcc.Store(id="free-click-coordinate"),
    dcc.Store(id="listener-ready"),
    # fires once, shortly after page load, to attach the JS click listener
    dcc.Interval(id="init-interval", interval=500, max_intervals=1),

    html.Div([
        html.Div(
            dcc.Graph(id="reduction-scatter", figure=fig),
            style={"width": "65%", "display": "inline-block", "verticalAlign": "top"}
        ),
        html.Div([
            html.H4("Nearest object"),
            html.Div(id="selected-object", children="Click anywhere on the plot"),
            html.H4("Coordinates"),
            html.Div(id="selected-coordinates"),
            html.H4("Reconstructed features"),
            html.Div(id="reconstructed-features"),
        ], style={"width": "30%", "display": "inline-block",
                  "verticalAlign": "top", "padding": "20px", 'fontSize': '8px'}),
    ])
])


# clientside callback
app.clientside_callback(
    """
    function(n) {
        const wrapper = document.getElementById("reduction-scatter");
        if (!wrapper || wrapper.dataset.freeClick) {
            return window.dash_clientside.no_update;
        }
        wrapper.dataset.freeClick = "1";

        let down = null;
        wrapper.addEventListener("mousedown", e => { down = [e.clientX, e.clientY]; });

        wrapper.addEventListener("click", function(e) {
            // ignore drags (zoom / pan)
            if (down && Math.hypot(e.clientX - down[0], e.clientY - down[1]) > 4) return;

            const gd   = wrapper.querySelector(".js-plotly-plot");
            const drag = wrapper.querySelector(".nsewdrag");   // the inner plot area
            if (!gd || !drag || !gd._fullLayout) return;

            const r = drag.getBoundingClientRect();
            if (e.clientX < r.left || e.clientX > r.right ||
                e.clientY < r.top  || e.clientY > r.bottom) return;   // outside axes

            const x = gd._fullLayout.xaxis.p2d(e.clientX - r.left);
            const y = gd._fullLayout.yaxis.p2d(e.clientY - r.top);

            window.dash_clientside.set_props(
                "free-click-coordinate",
                {data: {x: x, y: y, t: Date.now()}}
            );
        });
        return "ready";
    }
    """,
    Output("listener-ready", "data"),
    Input("init-interval", "n_intervals"),
)


# callback
@app.callback(
    Output("selected-object", "children"),
    Output("selected-coordinates", "children"),
    Output("reconstructed-features", "children"),
    Output("reduction-scatter", "figure"),
    Input("free-click-coordinate", "data"),
    prevent_initial_call=True,
)
def update_reconstruction(click):
    if not click:
        return no_update, no_update, no_update, no_update

    x, y = click["x"], click["y"]

    # nearest real object (just for info)
    d = (Z["component_1"] - x) ** 2 + (Z["component_2"] - y) ** 2
    nearest = Z.loc[d.idxmin(), "object"]

    # reconstruct original features
    #prediction = reconstruction_model.predict([[x, y]])[0]

    prediction = reconstruction_model.predict([[x,y]])[0]
    pred_df = pd.DataFrame(data=[prediction], columns=transformed.columns)
    reverse = pd.DataFrame(data=ct.named_transformers_['StandardScaler'].inverse_transform(pred_df[list(ct.named_transformers_['StandardScaler'].get_feature_names_out())]),    
                           columns=list(ct.named_transformers_['StandardScaler'].get_feature_names_out()))
   
    reverse_ohe = pd.DataFrame(data=ct.named_transformers_['OHE'].inverse_transform(transformed[list(ct.named_transformers_['OHE'].get_feature_names_out())]),
                           index=labels,
                           columns=col_ohe)

    
    #reverse[col_ohe[0]] = ct.named_transformers_['OHE'].inverse_transform(pred_df[list(ct.named_transformers_['OHE'].get_feature_names_out())]).reshape(pred_df.shape[0])
    
    for a in reverse_ohe.columns:
        reverse[a] = reverse_ohe[a]


    prediction = list(np.array(reverse[:1])[0])

    rows = [
        html.Tr([html.Td(name), html.Td(f"{value:.3f}")])
        for name, value in zip(reverse.columns, prediction)
    ]
    table = html.Table(
        [html.Thead(html.Tr([html.Th("Feature"), html.Th("Estimated value")])),
         html.Tbody(rows)],
        style={"width": "100%"}
    )

    # move the red marker without rebuilding the whole figure
    patched = Patch()
   #  patched["data"][1]["x"] = [x]
   #  patched["data"][1]["y"] = [y]
    patched["data"][MARKER_IDX]["x"] = [x]
    patched["data"][MARKER_IDX]["y"] = [y]

    return (
        nearest,
        html.Div([f"X = {x:.3f}", html.Br(), f"Y = {y:.3f}"]),
        table,
        patched,
    )


# start
if __name__ == "__main__":
    app.run(debug=False, port=8080)

/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/py

# 3.1 Alcohol products UMAP with restored categorical features after OHE
dimension reduction UMAP  
model Random Forest

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer
from dash import Dash, dcc, html, Input, Output, Patch, no_update
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
import umap


data = pd.read_excel('data/alcohol_products_table_r.xlsx', index_col=[0])
labels = data.index
col_ohe = ['Естественное брожение', 'Газированность', 'Исходный продукт']
sqrtran = FunctionTransformer(np.sqrt, inverse_func=np.square)
ct = ColumnTransformer(
    [("StandardScaler", StandardScaler(), data.columns.drop(['Естественное брожение', 'Газированность', 'Исходный продукт']).to_list()),
     ("OHE", OneHotEncoder(drop=None), ['Естественное брожение', 'Газированность', 'Исходный продукт'])], verbose_feature_names_out=False)

transformed = pd.DataFrame(data=ct.fit_transform(data), index=labels, columns=ct.get_feature_names_out())

reverse = pd.DataFrame(data=ct.named_transformers_['StandardScaler'].inverse_transform(transformed[list(ct.named_transformers_['StandardScaler'].get_feature_names_out())]),
                       index=labels,
                       columns=list(ct.named_transformers_['StandardScaler'].get_feature_names_out()))

reverse_ohe = pd.DataFrame(data=ct.named_transformers_['OHE'].inverse_transform(transformed[list(ct.named_transformers_['OHE'].get_feature_names_out())]),
                           index=labels,
                           columns=col_ohe)

for a in reverse_ohe.columns:
    reverse[a] = reverse_ohe[a]

dict_alk_groups = {'Красное вино':'вино',
                   'Белое вино':'вино',
                   'Молодое вино':'вино',
                   'Сидр':'пиво',
                   'Лагер':'пиво',
                   'ИПА':'пиво',
                   'Квас':'пиво',
                   'Самогон':'самогон',
                   'Чача':'самогон',
                   'Граппа':'самогон',
                   'Виски':'самогон',
                   'Текилла':'самогон',
                   'Водка':'водка',
                   'Аквавит':'водка',
                   'Кефир':'кефир'}

X_processed = transformed.copy()

reducer = umap.UMAP()
embedding = reducer.fit_transform(X_processed)


Z = pd.DataFrame(
    embedding,
    columns=["component_1", "component_2"]
)

Z["object"] = labels
Z['group'] = Z['object'].map(dict_alk_groups)

X_target = X_processed

reconstruction_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reconstruction_model.fit(
    Z[["component_1", "component_2"]],
    X_target
)



# dash part

# plot
fig = px.scatter(
    Z, x="component_1", y="component_2",
    color = 'group',
    text="object", hover_name="object", title="2D representation"
)
fig.update_traces(marker=dict(size=12), textfont_size=8)

MARKER_IDX = len(fig.data)

# extra trace (index 1) that shows where the user clicked
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(symbol="x", size=16, color="yellow"),
    showlegend=False, hoverinfo="skip"
))
fig.update_layout(uirevision="keep")   # keeps zoom/pan when the figure is patched


# dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H2("Dimension Reduction → Feature Reconstruction"),

    dcc.Store(id="free-click-coordinate"),
    dcc.Store(id="listener-ready"),
    # fires once, shortly after page load, to attach the JS click listener
    dcc.Interval(id="init-interval", interval=500, max_intervals=1),

    html.Div([
        html.Div(
            dcc.Graph(id="reduction-scatter", figure=fig),
            style={"width": "65%", "display": "inline-block", "verticalAlign": "top"}
        ),
        html.Div([
            html.H4("Nearest object"),
            html.Div(id="selected-object", children="Click anywhere on the plot"),
            html.H4("Coordinates"),
            html.Div(id="selected-coordinates"),
            html.H4("Reconstructed features"),
            html.Div(id="reconstructed-features"),
        ], style={"width": "30%", "display": "inline-block",
                  "verticalAlign": "top", "padding": "20px", 'fontSize': '8px'}),
    ])
])


# clientside callback
app.clientside_callback(
    """
    function(n) {
        const wrapper = document.getElementById("reduction-scatter");
        if (!wrapper || wrapper.dataset.freeClick) {
            return window.dash_clientside.no_update;
        }
        wrapper.dataset.freeClick = "1";

        let down = null;
        wrapper.addEventListener("mousedown", e => { down = [e.clientX, e.clientY]; });

        wrapper.addEventListener("click", function(e) {
            // ignore drags (zoom / pan)
            if (down && Math.hypot(e.clientX - down[0], e.clientY - down[1]) > 4) return;

            const gd   = wrapper.querySelector(".js-plotly-plot");
            const drag = wrapper.querySelector(".nsewdrag");   // the inner plot area
            if (!gd || !drag || !gd._fullLayout) return;

            const r = drag.getBoundingClientRect();
            if (e.clientX < r.left || e.clientX > r.right ||
                e.clientY < r.top  || e.clientY > r.bottom) return;   // outside axes

            const x = gd._fullLayout.xaxis.p2d(e.clientX - r.left);
            const y = gd._fullLayout.yaxis.p2d(e.clientY - r.top);

            window.dash_clientside.set_props(
                "free-click-coordinate",
                {data: {x: x, y: y, t: Date.now()}}
            );
        });
        return "ready";
    }
    """,
    Output("listener-ready", "data"),
    Input("init-interval", "n_intervals"),
)


# callback
@app.callback(
    Output("selected-object", "children"),
    Output("selected-coordinates", "children"),
    Output("reconstructed-features", "children"),
    Output("reduction-scatter", "figure"),
    Input("free-click-coordinate", "data"),
    prevent_initial_call=True,
)
def update_reconstruction(click):
    if not click:
        return no_update, no_update, no_update, no_update

    x, y = click["x"], click["y"]

    # nearest real object (just for info)
    d = (Z["component_1"] - x) ** 2 + (Z["component_2"] - y) ** 2
    nearest = Z.loc[d.idxmin(), "object"]

    # reconstruct original features

    prediction = reconstruction_model.predict([[x, y]])[0]
    pred_df = pd.DataFrame(data=[prediction], columns=transformed.columns)

    reverse = pd.DataFrame(
        data=ct.named_transformers_['StandardScaler'].inverse_transform(
            pred_df[list(ct.named_transformers_['StandardScaler'].get_feature_names_out())]
        ),
        columns=list(ct.named_transformers_['StandardScaler'].get_feature_names_out())
    )

    reverse_ohe = pd.DataFrame(
        data=ct.named_transformers_['OHE'].inverse_transform(
            pred_df[list(ct.named_transformers_['OHE'].get_feature_names_out())]
        ),
        columns=col_ohe
    )

    for a in reverse_ohe.columns:
        reverse[a] = reverse_ohe[a]

    prediction = list(np.array(reverse.iloc[0]))

    #print(reverse.columns, prediction)
    
    rows = [
        html.Tr([
            html.Td(name),
            html.Td(f"{value:.3f}" if isinstance(value, (int, float, np.floating, np.integer)) else str(value))
        ])
        for name, value in zip(reverse.columns, prediction)
    ]


    table = html.Table(
        [html.Thead(html.Tr([html.Th("Feature"), html.Th("Estimated value")])),
         html.Tbody(rows)],
        style={"width": "100%"}
    )

    # move the red marker without rebuilding the whole figure
    patched = Patch()
   #  patched["data"][1]["x"] = [x]
   #  patched["data"][1]["y"] = [y]
    patched["data"][MARKER_IDX]["x"] = [x]
    patched["data"][MARKER_IDX]["y"] = [y]

    return (
        nearest,
        html.Div([f"X = {x:.3f}", html.Br(), f"Y = {y:.3f}"]),
        table,
        patched,
    )


# start
if __name__ == "__main__":
    app.run(debug=False, port=8080)

/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/py

# 4. Benthos Denmark
dimension reduction UMAP  
model Random Forest  
using processed [dataset from Pangea](https://doi.pangaea.de/10.1594/PANGAEA.870461)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer
from dash import Dash, dcc, html, Input, Output, Patch, no_update
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor


df = pd.read_csv('data/benthic_data_dania.csv', index_col='station')
Z = pd.read_csv('data/ordination_and_support_data_dania.csv')

col_save = df.columns
sqrtran = FunctionTransformer(np.sqrt)
sqrtran = FunctionTransformer(np.sqrt, inverse_func=np.square)
transformed = sqrtran.fit_transform(df)
transformed.columns = col_save
reverse = sqrtran.inverse_transform(transformed)

X_processed = transformed.copy()

Z = Z.rename(columns={'X':'component_1','Y':'component_2', 'station':'object'})


X_target = X_processed

reconstruction_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

reconstruction_model.fit(
    Z[["component_1", "component_2"]],
    X_target
)



# plot
fig = px.scatter(
    Z, x="component_1", y="component_2",
    #text="object", 
    color = "community",
    hover_name="community", hover_data = ['object','community','cluster'], title="Ordination"
)
fig.update_traces(marker=dict(size=12))

MARKER_IDX = len(fig.data)

# extra trace (index 1) that shows where the user clicked
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(symbol="x", size=16, color="yellow"),
    showlegend=False, hoverinfo="skip"
))
fig.update_layout(uirevision="keep")   # keeps zoom/pan when the figure is patched


# dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H2("Restoration of features after dimension reduction throw UMAP, benthos Denmark"),

    dcc.Store(id="free-click-coordinate"),
    dcc.Store(id="listener-ready"),
    # fires once, shortly after page load, to attach the JS click listener
    dcc.Interval(id="init-interval", interval=500, max_intervals=1),

    html.Div([
        html.Div(
            dcc.Graph(id="reduction-scatter", figure=fig),
            style={"width": "65%", "display": "inline-block", "verticalAlign": "top"}
        ),
        html.Div([
            html.H4("Nearest object"),
            html.Div(id="selected-object", children="Click anywhere on the plot"),
            html.H4("Coordinates"),
            html.Div(id="selected-coordinates"),
            html.H4("Reconstructed features"),
            html.Div(id="reconstructed-features"),
        ], style={"width": "30%", "display": "inline-block",
                  "verticalAlign": "top", "padding": "10px",'fontSize': '8px'}),
    ])
])


# clientside callback
app.clientside_callback(
    """
    function(n) {
        const wrapper = document.getElementById("reduction-scatter");
        if (!wrapper || wrapper.dataset.freeClick) {
            return window.dash_clientside.no_update;
        }
        wrapper.dataset.freeClick = "1";

        let down = null;
        wrapper.addEventListener("mousedown", e => { down = [e.clientX, e.clientY]; });

        wrapper.addEventListener("click", function(e) {
            // ignore drags (zoom / pan)
            if (down && Math.hypot(e.clientX - down[0], e.clientY - down[1]) > 4) return;

            const gd   = wrapper.querySelector(".js-plotly-plot");
            const drag = wrapper.querySelector(".nsewdrag");   // the inner plot area
            if (!gd || !drag || !gd._fullLayout) return;

            const r = drag.getBoundingClientRect();
            if (e.clientX < r.left || e.clientX > r.right ||
                e.clientY < r.top  || e.clientY > r.bottom) return;   // outside axes

            const x = gd._fullLayout.xaxis.p2d(e.clientX - r.left);
            const y = gd._fullLayout.yaxis.p2d(e.clientY - r.top);

            window.dash_clientside.set_props(
                "free-click-coordinate",
                {data: {x: x, y: y, t: Date.now()}}
            );
        });
        return "ready";
    }
    """,
    Output("listener-ready", "data"),
    Input("init-interval", "n_intervals"),
)


# callback
@app.callback(
    Output("selected-object", "children"),
    Output("selected-coordinates", "children"),
    Output("reconstructed-features", "children"),
    Output("reduction-scatter", "figure"),
    Input("free-click-coordinate", "data"),
    prevent_initial_call=True,
)
def update_reconstruction(click):
    if not click:
        return no_update, no_update, no_update, no_update

    x, y = click["x"], click["y"]

    # nearest real object (just for info)
    d = (Z["component_1"] - x) ** 2 + (Z["component_2"] - y) ** 2
    nearest = Z.loc[d.idxmin(), "object"]

    # reconstruct original features
    #prediction = reconstruction_model.predict([[x, y]])[0]

    prediction = reconstruction_model.predict([[x,y]])[0]
    pred_df = pd.DataFrame(data=[prediction], columns=transformed.columns)
    reverse = sqrtran.inverse_transform(pred_df)
    prediction = np.array(reverse[:1])[0]


    pairs = list(zip(reverse.columns, prediction))

    # sort by value, numeric features first (by magnitude), text features after (alphabetically)
    def sort_key(item):
        _, value = item
        if isinstance(value, (int, float, np.floating, np.integer)) and not isinstance(value, bool) and not pd.isna(value):
            return (0, -value)   # numeric, descending
        return (1, str(value))   # text, alphabetical, grouped after numbers

    pairs.sort(key=sort_key)

    rows = [
        html.Tr([html.Td(name), html.Td(f"{value:.3f}" if isinstance(value, (int, float, np.floating, np.integer)) else str(value))])
        for name, value in pairs
        ]



    '''
    rows = [
        html.Tr([html.Td(name), html.Td(f"{value:.3f}")])
        for name, value in zip(reverse.columns, prediction)
    ]
    '''

    table = html.Table(
        [html.Thead(html.Tr([html.Th("Feature"), html.Th("Estimated value")])),
         html.Tbody(rows)],
        style={"width": "100%"}
    )

    # move the red marker without rebuilding the whole figure
    patched = Patch()
   #  patched["data"][1]["x"] = [x]
   #  patched["data"][1]["y"] = [y]
    patched["data"][MARKER_IDX]["x"] = [x]
    patched["data"][MARKER_IDX]["y"] = [y]

    return (
        nearest,
        html.Div([f"X = {x:.3f}", html.Br(), f"Y = {y:.3f}"]),
        table,
        patched,
    )


# start
if __name__ == "__main__":
    app.run(debug=False, port=8080)

/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/py

# 5. Benthos from PhD thesis Kara Sea 
dimension reduction - MDS  
model KNeighborsRegressor

In [9]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer
from dash import Dash, dcc, html, Input, Output, Patch, no_update
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
import umap
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
from sklearn.neighbors import KNeighborsRegressor


k12_r = pd.read_excel('data/Kara_2012_east_for_2025_pres.xlsx', sheet_name='R_east',index_col=[0]).T
k12_comm = pd.read_excel('data/Kara_2012_east_for_2025_pres.xlsx', sheet_name='comm',index_col=[0]).T

'''
reducer = umap.UMAP()
embedding = reducer.fit_transform(k12_r)
embedding.shape
ordination_umap = pd.DataFrame(embedding, columns=['X', 'Y'])
ordination_umap['community'] = list(k12_comm['community'])
ordination_umap['station'] = list(k12_comm.index)
'''


col_save = k12_r.columns
#sqrtran = FunctionTransformer(np.sqrt)
sqrtran = FunctionTransformer(np.sqrt, inverse_func=np.square)
transformed = sqrtran.fit_transform(k12_r)
transformed.columns = col_save
reverse = sqrtran.inverse_transform(transformed)

embedding = MDS(n_components=2, dissimilarity='precomputed')
table_for_mds = embedding.fit_transform(squareform(pdist(transformed, metric='braycurtis')))
table_for_mds = pd.DataFrame(table_for_mds)
table_for_mds.columns = ['X', 'Y']
table_for_mds['community'] = list(k12_comm['community'])
table_for_mds['station'] = list(k12_comm.index)

Z = table_for_mds

X_processed = transformed.copy()


Z = Z.rename(columns={'X':'component_1','Y':'component_2', 'station':'object'})


X_target = X_processed

'''
reconstruction_model = RandomForestRegressor(
    n_estimators=200, 
    max_depth = 300,
    random_state=42
)
'''

reconstruction_model = KNeighborsRegressor(
    n_neighbors=5, 
    weights='uniform', 
    algorithm='auto', 
    leaf_size=30, 
    p=2, 
    metric='minkowski'
)


reconstruction_model.fit(
    Z[["component_1", "component_2"]],
    X_target
)



# plot
fig = px.scatter(
    Z, x="component_1", y="component_2",
    #text="object", 
    color = "community",
    hover_name="community", hover_data = ['object','community'], title="Ordination"
)
fig.update_traces(marker=dict(size=12))

MARKER_IDX = len(fig.data)

# extra trace (index 1) that shows where the user clicked
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(symbol="x", size=16, color="yellow"),
    showlegend=False, hoverinfo="skip"
))
fig.update_layout(uirevision="keep")   # keeps zoom/pan when the figure is patched


# dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H2("Restoration of features after dimension reduction throw MDS, benthos, Kara Sea"),

    dcc.Store(id="free-click-coordinate"),
    dcc.Store(id="listener-ready"),
    # fires once, shortly after page load, to attach the JS click listener
    dcc.Interval(id="init-interval", interval=500, max_intervals=1),

    html.Div([
        html.Div(
            dcc.Graph(id="reduction-scatter", figure=fig),
            style={"width": "65%", "display": "inline-block", "verticalAlign": "top"}
        ),
        html.Div([
            html.H4("Nearest object"),
            html.Div(id="selected-object", children="Click anywhere on the plot"),
            html.H4("Coordinates"),
            html.Div(id="selected-coordinates"),
            html.H4("Reconstructed features"),
            html.Div(id="reconstructed-features"),
        ], style={"width": "30%", "display": "inline-block",
                  "verticalAlign": "top", "padding": "10px",'fontSize': '8px'}),
    ])
])


# clientside callback
app.clientside_callback(
    """
    function(n) {
        const wrapper = document.getElementById("reduction-scatter");
        if (!wrapper || wrapper.dataset.freeClick) {
            return window.dash_clientside.no_update;
        }
        wrapper.dataset.freeClick = "1";

        let down = null;
        wrapper.addEventListener("mousedown", e => { down = [e.clientX, e.clientY]; });

        wrapper.addEventListener("click", function(e) {
            // ignore drags (zoom / pan)
            if (down && Math.hypot(e.clientX - down[0], e.clientY - down[1]) > 4) return;

            const gd   = wrapper.querySelector(".js-plotly-plot");
            const drag = wrapper.querySelector(".nsewdrag");   // the inner plot area
            if (!gd || !drag || !gd._fullLayout) return;

            const r = drag.getBoundingClientRect();
            if (e.clientX < r.left || e.clientX > r.right ||
                e.clientY < r.top  || e.clientY > r.bottom) return;   // outside axes

            const x = gd._fullLayout.xaxis.p2d(e.clientX - r.left);
            const y = gd._fullLayout.yaxis.p2d(e.clientY - r.top);

            window.dash_clientside.set_props(
                "free-click-coordinate",
                {data: {x: x, y: y, t: Date.now()}}
            );
        });
        return "ready";
    }
    """,
    Output("listener-ready", "data"),
    Input("init-interval", "n_intervals"),
)


# callback
@app.callback(
    Output("selected-object", "children"),
    Output("selected-coordinates", "children"),
    Output("reconstructed-features", "children"),
    Output("reduction-scatter", "figure"),
    Input("free-click-coordinate", "data"),
    prevent_initial_call=True,
)
def update_reconstruction(click):
    if not click:
        return no_update, no_update, no_update, no_update

    x, y = click["x"], click["y"]

    # nearest real object (just for info)
    d = (Z["component_1"] - x) ** 2 + (Z["component_2"] - y) ** 2
    nearest = Z.loc[d.idxmin(), "object"]

    # reconstruct original features
    #prediction = reconstruction_model.predict([[x, y]])[0]

    prediction = reconstruction_model.predict([[x,y]])[0]
    pred_df = pd.DataFrame(data=[prediction], columns=transformed.columns)
    reverse = sqrtran.inverse_transform(pred_df)
    prediction = np.array(reverse[:1])[0]

    pairs = list(zip(reverse.columns, prediction))

    # sort by value, numeric features first (by magnitude), text features after (alphabetically)
    def sort_key(item):
        _, value = item
        if isinstance(value, (int, float, np.floating, np.integer)) and not isinstance(value, bool) and not pd.isna(value):
            return (0, -value)   # numeric, descending
        return (1, str(value))   # text, alphabetical, grouped after numbers

    pairs.sort(key=sort_key)

    rows = [
        html.Tr([html.Td(name), html.Td(f"{value:.3f}" if isinstance(value, (int, float, np.floating, np.integer)) else str(value))])
        for name, value in pairs
        ]

    '''
    rows = [
        html.Tr([html.Td(name), html.Td(f"{value:.3f}")])
        for name, value in zip(reverse.columns, prediction)
    ]
    '''


    table = html.Table(
        [html.Thead(html.Tr([html.Th("Feature"), html.Th("Estimated value")])),
         html.Tbody(rows)],
        style={"width": "100%"}
    )

    # move the red marker without rebuilding the whole figure
    patched = Patch()
   #  patched["data"][1]["x"] = [x]
   #  patched["data"][1]["y"] = [y]
    patched["data"][MARKER_IDX]["x"] = [x]
    patched["data"][MARKER_IDX]["y"] = [y]

    return (
        nearest,
        html.Div([f"X = {x:.3f}", html.Br(), f"Y = {y:.3f}"]),
        table,
        patched,
    )


# start
if __name__ == "__main__":
    app.run(debug=False, port=8080)

/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
/home/vlad/some_projects/dimred_dash/.dimred_venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
